In [ ]:
# краткая документация по назначению ноутбука
"""
назначение: объединенная генерация pattern-признаков для elliptic data set.
основные шаги: загрузка исходных csv, расчет fan-in/fan-out, peel chains,
burst patterns, объединение новых признаков с исходной таблицей транзакций,
сохранение расширенного датасета для последующей загрузки в kaggle.
зависимости и данные: pandas, numpy, networkx, sklearn, matplotlib, seaborn,
опционально torch; elliptic_txs_features.csv, elliptic_txs_classes.csv,
elliptic_txs_edgelist.csv.
ключевые переменные:
- nodes_df: основная таблица txid, классов, time_step и исходных признаков.
- fan_features: hub-признаки fan-in/fan-out для каждого txid.
- peel_features: признаки участия txid в peel chains и их anomaly score.
- burst_features: временные burst-признаки активности по txid и time_step.
- enriched_df: итоговая таблица после left join всех групп признаков.
принцип функций:
- нормализация score делает признаки разных паттернов сопоставимыми.
- fan-блок использует степени и hub-score, возвращает node-level признаки.
- peel-блок строит цепочки, оценивает их и агрегирует chain-level score к узлам.
- burst-блок считает временную активность и возвращает ranking-признаки.
гипотезы:
- pattern-признаки добавляют структурный сигнал поверх анонимизированных elliptic features.
- отсутствие паттерна кодируется нулем после join, потому что узел не участвовал в найденном событии.
- единый enriched dataset нужен для честного ablation без повторного пересчета паттернов.
"""

from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.preprocessing import MinMaxScaler, StandardScaler

try:
    import torch
except ImportError:
    torch = None

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") if torch else "cpu"

print(f"Device: {DEVICE}")
sns.set_theme(style="whitegrid")

# Объединение AML-паттернов и обогащение Elliptic Data Set

Цель ноутбука: собрать признаки из трёх pattern-направлений и сохранить расширенную версию исходного Elliptic Data Set.

Разделы:

1. Fan-in / Fan-out.
2. Peel Chains.
3. Burst Patterns.
4. Подготовка дополнительного feature dataframe и сохранение enriched dataset.

Этот ноутбук не заменяет отдельные исследовательские ноутбуки `04`, `05`, `06`. Он нужен как единый pipeline для генерации признаков.

## Загрузка данных

Используются стандартные файлы Elliptic:

- `elliptic_txs_features.csv`;
- `elliptic_txs_classes.csv`;
- `elliptic_txs_edgelist.csv`.

В Kaggle основной путь обычно такой:

`/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset`

In [ ]:
# задаем пути к данным и папке вывода
PROJECT_ROOT = Path.cwd()
DATA_CANDIDATES = [
    Path(
        "/kaggle/input/datasets/organizations/ellipticco/"
        "elliptic-data-set/elliptic_bitcoin_dataset"
    ),
    Path("/kaggle/input/elliptic-data-set/elliptic_bitcoin_dataset"),
    PROJECT_ROOT / "elliptic_bitcoin_dataset",
    PROJECT_ROOT / "data" / "elliptic_bitcoin_dataset",
    PROJECT_ROOT.parent / "input" / "elliptic-data-set" / "elliptic_bitcoin_dataset",
]


# вход: список возможных папок; выход: директория, где найдены все обязательные csv elliptic.
def find_data_dir(candidates):
    required = {
        "elliptic_txs_features.csv",
        "elliptic_txs_classes.csv",
        "elliptic_txs_edgelist.csv",
    }
    checked = []
    for candidate in candidates:
        checked.append(str(candidate))
        if candidate.exists():
            names = {path.name for path in candidate.iterdir()}
            if required.issubset(names):
                return candidate
    checked_paths = "\n".join(checked)
    raise FileNotFoundError(
        "Elliptic CSV files were not found. Checked paths:\n"
        f"{checked_paths}"
    )


DATA_DIR = find_data_dir(DATA_CANDIDATES)
FEATURES_PATH = DATA_DIR / "elliptic_txs_features.csv"
CLASSES_PATH = DATA_DIR / "elliptic_txs_classes.csv"
EDGES_PATH = DATA_DIR / "elliptic_txs_edgelist.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "elliptic_enriched_dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# загружаем исходные таблицы Elliptic и собираем общий dataframe узлов

# вход: пути к features, classes и edgelist; выход: исходные таблицы и объединенная таблица nodes.
def load_elliptic(features_path, classes_path, edges_path):
    features = pd.read_csv(features_path, header=None)
    feature_columns = ["txId", "time_step"] + [
        f"feature_{idx:03d}" for idx in range(features.shape[1] - 2)
    ]
    features.columns = feature_columns

    classes = pd.read_csv(classes_path)
    edges = pd.read_csv(edges_path)

    classes["class"] = classes["class"].astype(str)
    classes["label"] = classes["class"].map({"1": 1, "2": 0})
    classes["class_name"] = classes["class"].map(
        {"1": "illicit", "2": "licit", "unknown": "unknown"}
    ).fillna("unknown")

    nodes = features.merge(classes, on="txId", how="left")
    return features, classes, edges, nodes


features_df, classes_df, edges_df, nodes_df = load_elliptic(
    FEATURES_PATH,
    CLASSES_PATH,
    EDGES_PATH,
)

# G хранит directed-граф транзакций, общий для fan, peel и burst feature engineering.
G = nx.from_pandas_edgelist(
    edges_df,
    source="txId1",
    target="txId2",
    create_using=nx.DiGraph(),
)

# node_time и node_label ускоряют доступ к времени и классу без повторных merge.
node_time = nodes_df.set_index("txId")["time_step"].to_dict()
node_label = nodes_df.set_index("txId")["label"].to_dict()
feature_cols = [col for col in features_df.columns if col.startswith("feature_")]

print(f"Features: {features_df.shape}")
print(f"Classes: {classes_df.shape}")
print(f"Edges: {edges_df.shape}")
print(f"Graph nodes: {G.number_of_nodes()}")
print(f"Graph edges: {G.number_of_edges()}")
print(f"Time steps: {nodes_df['time_step'].nunique()}")
print(f"Labeled nodes: {nodes_df['label'].notna().sum()}")

In [ ]:
# общие вспомогательные функции для нормализации и оценки ранжирования

# вход: числовой массив; выход: устойчивый z-score, менее чувствительный к выбросам.
def robust_zscore(values):
    values = np.asarray(values, dtype=float)
    median = np.nanmedian(values)
    mad = np.nanmedian(np.abs(values - median))
    if mad == 0 or np.isnan(mad):
        std = np.nanstd(values)
        if std == 0 or np.isnan(std):
            return np.zeros_like(values, dtype=float)
        return (values - np.nanmean(values)) / std
    return 0.6745 * (values - median) / mad


def minmax(series):
    values = pd.Series(series).fillna(0).to_numpy().reshape(-1, 1)
    return MinMaxScaler().fit_transform(values).ravel()


# вход: dataframe и score; выход: precision/recall@k и average precision на размеченных узлах.
def evaluate_ranking(frame, score_col, label_col="label", top_fracs=(0.001, 0.005, 0.01, 0.05)):
    labeled = frame.dropna(subset=[label_col]).copy()
    labeled[label_col] = labeled[label_col].astype(int)
    labeled = labeled.sort_values(score_col, ascending=False)

    rows = []
    positives = labeled[label_col].sum()
    for frac in top_fracs:
        k = max(1, int(len(labeled) * frac))
        top_k = labeled.head(k)
        true_positive = top_k[label_col].sum()
        rows.append({
            "score": score_col,
            "top_frac": frac,
            "top_k": k,
            "precision_at_k": true_positive / k,
            "recall_at_k": true_positive / positives if positives else np.nan,
            "illicit_found": int(true_positive),
        })

    rows_df = pd.DataFrame(rows)
    rows_df["average_precision"] = average_precision_score(
        labeled[label_col],
        labeled[score_col],
    )
    return rows_df

# 1. Fan-in / Fan-out

Паттерн ищет транзакции с аномально большим числом входящих или исходящих связей.

Подходы из `04-elliptic-fan-in-fan-out.ipynb`:

- `in_degree / out_degree`;
- z-score по степеням;
- rule-based hub labels;
- FlowScope-inspired локальный `anomalousness_score`;
- признаки для последующей классификации узлов.

In [ ]:
# считаем hub-признаки для fan-in и fan-out паттернов

# вход: граф и узлы; выход: node-level признаки fan-in, fan-out и bipartite hub.
def build_fan_features(graph, nodes):
    degree_frame = pd.DataFrame({
        "txId": nodes["txId"],
    })
    degree_frame["in_degree"] = degree_frame["txId"].map(
        lambda node: graph.in_degree(node) if node in graph else 0
    )
    degree_frame["out_degree"] = degree_frame["txId"].map(
        lambda node: graph.out_degree(node) if node in graph else 0
    )

    degree_frame["z_in"] = robust_zscore(degree_frame["in_degree"])
    degree_frame["z_out"] = robust_zscore(degree_frame["out_degree"])

    # пороги берут верхний хвост степеней и z-score, чтобы hub был редким отклонением, а не обычной вершиной.
    fanin_threshold = max(3, int(degree_frame["in_degree"].quantile(0.99)))
    fanout_threshold = max(3, int(degree_frame["out_degree"].quantile(0.99)))
    z_threshold = 3.0

    degree_frame["is_fanin_hub"] = (
        (degree_frame["in_degree"] >= fanin_threshold)
        & (degree_frame["z_in"] >= z_threshold)
    ).astype(int)
    degree_frame["is_fanout_hub"] = (
        (degree_frame["out_degree"] >= fanout_threshold)
        & (degree_frame["z_out"] >= z_threshold)
    ).astype(int)
    degree_frame["is_bipartite_hub"] = (
        (degree_frame["is_fanin_hub"] == 1)
        & (degree_frame["is_fanout_hub"] == 1)
    ).astype(int)

    conditions = [
        degree_frame["is_bipartite_hub"] == 1,
        degree_frame["is_fanin_hub"] == 1,
        degree_frame["is_fanout_hub"] == 1,
    ]
    choices = ["bipartite", "fanin", "fanout"]
    degree_frame["hub_type"] = np.select(conditions, choices, default="normal")

    # raw_scores отражает силу fan-паттерна через совместный размер входящего и исходящего фронта.
    raw_scores = []
    for node in degree_frame["txId"]:
        if node not in graph:
            raw_scores.append(0.0)
            continue
        predecessors = list(graph.predecessors(node))
        successors = list(graph.successors(node))
        n_pred = len(predecessors)
        n_succ = len(successors)
        if n_pred == 0 and n_succ == 0:
            raw_scores.append(0.0)
            continue
        balance = np.sqrt(max(n_pred, 1) * max(n_succ, 1))
        raw_scores.append((n_pred + n_succ) / balance)

    degree_frame["anomalousness_raw"] = raw_scores
    degree_frame["anomalousness_score"] = minmax(degree_frame["anomalousness_raw"])
    degree_frame["elliptic_time_step_feature"] = nodes["time_step"].to_numpy()

    return degree_frame.drop(columns=["anomalousness_raw"])


fan_features = build_fan_features(G, nodes_df)

fan_summary = fan_features["hub_type"].value_counts().rename_axis("hub_type").reset_index(name="count")
fan_summary

In [ ]:
# оцениваем fan-score на размеченных узлах и строим простые графики
fan_eval = fan_features.merge(nodes_df[["txId", "label"]], on="txId", how="left")
fan_ranking_metrics = evaluate_ranking(fan_eval, "anomalousness_score")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=fan_features, x="hub_type", ax=axes[0])
axes[0].set_title("Fan-in / Fan-out hub types")
axes[0].set_xlabel("hub_type")
axes[0].set_ylabel("count")

sns.histplot(fan_features["anomalousness_score"], bins=60, ax=axes[1])
axes[1].set_title("Fan anomalousness score")
axes[1].set_xlabel("anomalousness_score")

plt.tight_layout()
plt.show()

fan_ranking_metrics

# 2. Peel Chains

Паттерн ищет цепочки слоения: последовательные переходы по directed-графу, где один путь продолжается как main chain, а боковые ветви интерпретируются как peel-like активность.

В объединённом ноутбуке используется практичная версия из `05-elliptic-peel-chains.ipynb`:

- rule-based поиск цепочек;
- агрегаты по цепочке;
- GRU autoencoder для sequence anomaly score;
- Isolation Forest, LOF, PCA reconstruction для табличной аномальности;
- node-level признаки для последующего merge с Elliptic.

In [ ]:
# задаем параметры rule-based поиска peel chains
@dataclass
class PeelParams:
    min_length: int = 3
    max_length: int = 18
    max_start_nodes: int = 12000
    max_successors_per_step: int = 12
    max_delta_t: int = 6


# вход: текущий узел цепочки; выход: лучший следующий узел с учетом времени и степеней.
def select_next_chain_node(graph, current, visited, node_time_map, in_degree_map, out_degree_map):
    # выбираем следующий узел цепочки среди непосещенных successors
    successors = [node for node in graph.successors(current) if node not in visited]
    if not successors:
        return None

    current_time = node_time_map.get(current, 0)
    # valid_successors запрещает движение назад во времени, чтобы цепочка была temporal-consistent.
    valid_successors = []
    for node in successors:
        next_time = node_time_map.get(node, current_time)
        if next_time >= current_time:
            valid_successors.append(node)

    if not valid_successors:
        return None

    return max(
        valid_successors,
        key=lambda node: (
            out_degree_map.get(node, 0),
            in_degree_map.get(node, 0),
            -abs(node_time_map.get(node, current_time) - current_time),
        ),
    )


# вход: граф и таблица узлов; выход: список найденных temporal peel-like цепочек.
def find_peel_chains(graph, nodes, node_time_map, params=PeelParams(), log_every=1000):
    # заранее считаем степени, чтобы не дергать graph.degree в больших циклах
    in_degree_map = dict(graph.in_degree())
    out_degree_map = dict(graph.out_degree())

    # candidate_starts ограничивает поиск наиболее ветвящимися ранними узлами, где peel-поведение вероятнее.
    candidate_starts = (
        nodes.assign(out_degree=nodes["txId"].map(lambda node: out_degree_map.get(node, 0)))
        .query("out_degree > 0")
        .sort_values(["out_degree", "time_step"], ascending=[False, True])
        .head(params.max_start_nodes)["txId"]
        .tolist()
    )

    print(f"Start peel chain search: {len(candidate_starts)} start nodes")

    chains = []
    seen_paths = set()
    for start_idx, start in enumerate(candidate_starts):
        if start_idx % log_every == 0:
            print(f"Processed start nodes: {start_idx}/{len(candidate_starts)}, chains: {len(chains)}")

        chain = [start]
        visited = {start}
        current = start

        for _ in range(params.max_length - 1):
            next_node = select_next_chain_node(
                graph,
                current,
                visited,
                node_time_map,
                in_degree_map,
                out_degree_map,
            )
            if next_node is None:
                break
            delta_t = node_time_map.get(next_node, 0) - node_time_map.get(current, 0)
            if delta_t < 0 or delta_t > params.max_delta_t:
                break
            chain.append(next_node)
            visited.add(next_node)
            current = next_node

        if len(chain) >= params.min_length:
            path_key = tuple(chain)
            if path_key not in seen_paths:
                seen_paths.add(path_key)
                chains.append(chain)

    print(f"Processed start nodes: {len(candidate_starts)}/{len(candidate_starts)}, chains: {len(chains)}")
    return chains


peel_chains = find_peel_chains(G, nodes_df, node_time, PeelParams())
print(f"Peel chains found: {len(peel_chains)}")
print(f"Example chain length: {len(peel_chains[0]) if peel_chains else 0}")

In [ ]:
# считаем признаки цепочек с кэшем, чтобы не пересоздавать индекс nodes на каждой цепочке

# вход: одна цепочка txId; выход: агрегированные chain-level признаки времени, степеней и base features.
def chain_to_features(chain, node_table, in_degree_map, out_degree_map, feature_columns):
    # берем строки узлов цепочки из заранее подготовленного индекса
    node_rows = node_table.loc[chain]
    times = node_rows["time_step"].to_numpy(dtype=float)
    local_features = node_rows[feature_columns[:10]].to_numpy(dtype=float)

    in_degrees = [in_degree_map.get(node, 0) for node in chain]
    out_degrees = [out_degree_map.get(node, 0) for node in chain]

    return {
        "length": len(chain),
        "time_span": float(times.max() - times.min()) if len(times) else 0.0,
        "mean_out_degree": float(np.mean(out_degrees)),
        "mean_in_degree": float(np.mean(in_degrees)),
        "max_out_degree": float(np.max(out_degrees)),
        "feature_abs_mean": float(np.mean(np.abs(local_features))),
        "feature_abs_std": float(np.std(np.abs(local_features))),
    }


# вход: список цепочек; выход: таблица признаков для anomaly scoring и агрегации в node-level.
def build_chain_feature_frame(chains, graph, nodes, feature_columns, log_every=500):
    # готовим кэши один раз перед циклом
    node_table = nodes.set_index("txId")
    in_degree_map = dict(graph.in_degree())
    out_degree_map = dict(graph.out_degree())

    rows = []
    total_chains = len(chains)
    print(f"Start chain feature extraction: {total_chains} chains")

    for chain_id, chain in enumerate(chains):
        if chain_id % log_every == 0:
            print(f"Processed chains: {chain_id}/{total_chains}")

        row = {"chain_id": chain_id, "nodes": chain}
        row.update(
            chain_to_features(
                chain,
                node_table,
                in_degree_map,
                out_degree_map,
                feature_columns,
            )
        )
        rows.append(row)

    print(f"Processed chains: {total_chains}/{total_chains}")
    return pd.DataFrame(rows)


peel_chain_scores = build_chain_feature_frame(peel_chains, G, nodes_df, feature_cols)
peel_chain_scores.head()

In [ ]:
# готовим тензор цепочек для GRU autoencoder

# вход: цепочки и узловые признаки; выход: padded tensor для GRU autoencoder.
def build_chain_tensor(chains, nodes, feature_columns, max_length=18):
    if not chains:
        return np.zeros((0, max_length, 5), dtype=np.float32)

    node_table = nodes.set_index("txId")
    # tensor хранит компактные признаки каждого шага цепочки: время, степени и первые base features.
    tensor = np.zeros((len(chains), max_length, 5), dtype=np.float32)
    for chain_idx, chain in enumerate(chains):
        for step_idx, node in enumerate(chain[:max_length]):
            row = node_table.loc[node]
            tensor[chain_idx, step_idx, 0] = row["time_step"]
            tensor[chain_idx, step_idx, 1] = G.in_degree(node) if node in G else 0
            tensor[chain_idx, step_idx, 2] = G.out_degree(node) if node in G else 0
            tensor[chain_idx, step_idx, 3] = np.abs(row[feature_columns[:10]]).mean()
            tensor[chain_idx, step_idx, 4] = step_idx / max(max_length - 1, 1)

    flat = tensor.reshape(-1, tensor.shape[-1])
    flat = StandardScaler().fit_transform(flat)
    return flat.reshape(tensor.shape).astype(np.float32)


sequence_tensor = build_chain_tensor(peel_chains, nodes_df, feature_cols, max_length=18)
print(f"Sequence tensor shape: {sequence_tensor.shape}")

if torch is None or len(sequence_tensor) == 0:
    peel_chain_scores["ae_error"] = 0.0
    print("Torch is not installed or no chains found, AE score set to 0")
else:
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    class SequenceAutoencoder(nn.Module):
        def __init__(self, input_size=5, hidden_size=32):
            super().__init__()
            self.encoder = nn.GRU(input_size, hidden_size, batch_first=True)
            self.decoder = nn.GRU(input_size, hidden_size, batch_first=True)
            self.head = nn.Linear(hidden_size, input_size)

        def forward(self, x):
            _, hidden = self.encoder(x)
            output, _ = self.decoder(x, hidden)
            return self.head(output)

    x_tensor = torch.tensor(sequence_tensor, dtype=torch.float32)
    loader = DataLoader(TensorDataset(x_tensor), batch_size=64, shuffle=True)
    model = SequenceAutoencoder(input_size=sequence_tensor.shape[-1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(25):
        losses = []
        for (batch_x,) in loader:
            batch_x = batch_x.to(DEVICE)
            optimizer.zero_grad()
            reconstruction = model(batch_x)
            loss = loss_fn(reconstruction, batch_x)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if epoch in {0, 9, 24}:
            print(f"AE epoch {epoch + 1}: loss {np.mean(losses):.6f}")

    model.eval()
    with torch.no_grad():
        reconstructed = model(x_tensor.to(DEVICE)).cpu().numpy()
    ae_error = ((reconstructed - sequence_tensor) ** 2).mean(axis=(1, 2))
    peel_chain_scores["ae_error"] = ae_error

In [ ]:
# добавляем anomaly scores для найденных peel chains

# вход: chain-level признаки; выход: те же цепочки с tabular, neighborhood и ensemble anomaly-score.
def add_chain_anomaly_scores(chain_scores):
    if len(chain_scores) == 0:
        return chain_scores.assign(
            tab_anom_score=pd.Series(dtype=float),
            gnn_anom_score=pd.Series(dtype=float),
            ensemble_rank_score=pd.Series(dtype=float),
        )

    # score_columns задает признаки цепочки, по которым ищутся аномальные peel-паттерны.
    score_columns = [
        "length",
        "time_span",
        "mean_out_degree",
        "mean_in_degree",
        "max_out_degree",
        "feature_abs_mean",
        "feature_abs_std",
    ]
    x = chain_scores[score_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
    x_scaled = StandardScaler().fit_transform(x)

    if len(chain_scores) >= 10:
        iforest_score = -IsolationForest(
            n_estimators=200,
            contamination=0.05,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ).fit(x_scaled).decision_function(x_scaled)

        lof_neighbors = min(20, max(2, len(chain_scores) - 1))
        lof_score = -LocalOutlierFactor(n_neighbors=lof_neighbors).fit_predict(x_scaled)
        pca_components = min(3, x_scaled.shape[1], len(chain_scores))
        pca = PCA(n_components=pca_components, random_state=RANDOM_STATE)
        projected = pca.fit_transform(x_scaled)
        reconstructed = pca.inverse_transform(projected)
        pca_error = ((x_scaled - reconstructed) ** 2).mean(axis=1)

        # tab_anom_score усредняет несколько unsupervised сигналов, чтобы снизить зависимость от одного метода.
        chain_scores["tab_anom_score"] = (
            pd.Series(iforest_score).rank(pct=True).to_numpy()
            + pd.Series(lof_score).rank(pct=True).to_numpy()
            + pd.Series(pca_error).rank(pct=True).to_numpy()
        ) / 3

        embedding = projected
        nn_count = min(5, len(chain_scores))
        distances, _ = NearestNeighbors(n_neighbors=nn_count).fit(embedding).kneighbors(embedding)
        local_density_score = distances.mean(axis=1)
        chain_scores["gnn_anom_score"] = pd.Series(local_density_score).rank(pct=True).to_numpy()
    else:
        chain_scores["tab_anom_score"] = 0.0
        chain_scores["gnn_anom_score"] = 0.0

    chain_scores["ensemble_rank_score"] = (
        chain_scores["ae_error"].rank(pct=True)
        + chain_scores["tab_anom_score"].rank(pct=True)
        + chain_scores["gnn_anom_score"].rank(pct=True)
    ) / 3
    return chain_scores


peel_chain_scores = add_chain_anomaly_scores(peel_chain_scores)
peel_chain_scores.sort_values("ensemble_rank_score", ascending=False).head()

In [ ]:
# агрегируем chain-level peel scores в node-level признаки

# вход: цепочки и их score; выход: node-level peel-признаки для объединения с исходными features.
def build_node_peel_features(nodes, chains, chain_scores):
    node_stats = defaultdict(lambda: {
        "peel_in_chain_any": 0,
        "peel_chain_count": 0,
        "peel_baseline_chain_participation": 0,
        "peel_start_count": 0,
        "peel_middle_count": 0,
        "peel_end_count": 0,
        "peel_max_chain_length": 0,
        "ae_errors": [],
        "tab_scores": [],
        "gnn_scores": [],
    })

    # score_lookup связывает chain_id с anomaly-score, чтобы перенести сигнал на каждый txId цепочки.
    score_lookup = chain_scores.set_index("chain_id").to_dict("index") if len(chain_scores) else {}

    for chain_id, chain in enumerate(chains):
        scores = score_lookup.get(chain_id, {})
        for position, node in enumerate(chain):
            stats = node_stats[node]
            stats["peel_in_chain_any"] = 1
            stats["peel_chain_count"] += 1
            stats["peel_baseline_chain_participation"] = 1
            stats["peel_max_chain_length"] = max(stats["peel_max_chain_length"], len(chain))
            stats["ae_errors"].append(scores.get("ae_error", 0.0))
            stats["tab_scores"].append(scores.get("tab_anom_score", 0.0))
            stats["gnn_scores"].append(scores.get("gnn_anom_score", 0.0))

            if position == 0:
                stats["peel_start_count"] += 1
            elif position == len(chain) - 1:
                stats["peel_end_count"] += 1
            else:
                stats["peel_middle_count"] += 1

    rows = []
    for tx_id in nodes["txId"]:
        stats = node_stats[tx_id]
        rows.append({
            "txId": tx_id,
            "peel_in_chain_any": stats["peel_in_chain_any"],
            "peel_chain_count": stats["peel_chain_count"],
            "peel_baseline_chain_participation": stats["peel_baseline_chain_participation"],
            "peel_start_count": stats["peel_start_count"],
            "peel_middle_count": stats["peel_middle_count"],
            "peel_end_count": stats["peel_end_count"],
            "peel_max_chain_length": stats["peel_max_chain_length"],
            "peel_mean_ae_error": np.mean(stats["ae_errors"]) if stats["ae_errors"] else 0.0,
            "peel_max_ae_error": np.max(stats["ae_errors"]) if stats["ae_errors"] else 0.0,
            "peel_mean_tab_anom_score": np.mean(stats["tab_scores"]) if stats["tab_scores"] else 0.0,
            "peel_mean_gnn_anom_score": np.mean(stats["gnn_scores"]) if stats["gnn_scores"] else 0.0,
        })
    return pd.DataFrame(rows)


peel_features = build_node_peel_features(nodes_df, peel_chains, peel_chain_scores)
peel_features.head()

# 3. Burst Patterns

Паттерн ищет резкие всплески активности узла во временном графе.

Подходы из `06-elliptic-burst-patterns.ipynb`:

- построение временных событий по ребрам;
- `in_events`, `out_events`, `total_events`;
- robust z-score внутри каждого `time_step`;
- rule-based `burst_score_rule`;
- Isolation Forest как unsupervised anomaly detector;
- опциональный GRU forecasting error на `DEVICE`.

In [ ]:
# строим временные события по ребрам графа

# вход: edgelist и время узлов; выход: таблица входящих/исходящих событий по txId и timestep.
def build_temporal_edge_events(edges, node_time_map):
    edge_events = edges.copy()
    edge_events["src_time"] = edge_events["txId1"].map(node_time_map)
    edge_events["dst_time"] = edge_events["txId2"].map(node_time_map)
    edge_events = edge_events.dropna(subset=["src_time", "dst_time"])
    # event_time задается поздним концом ребра, чтобы связь не появлялась до времени целевого узла.
    edge_events["event_time"] = edge_events[["src_time", "dst_time"]].max(axis=1)
    edge_events["event_time"] = edge_events["event_time"].astype(int)

    src_events = edge_events[["txId1", "event_time"]].rename(columns={"txId1": "txId"})
    src_events["out_events"] = 1
    src_events["in_events"] = 0

    dst_events = edge_events[["txId2", "event_time"]].rename(columns={"txId2": "txId"})
    dst_events["out_events"] = 0
    dst_events["in_events"] = 1

    events = pd.concat([src_events, dst_events], ignore_index=True)
    activity = events.groupby(["txId", "event_time"], as_index=False).agg(
        in_events=("in_events", "sum"),
        out_events=("out_events", "sum"),
    )
    activity["total_events"] = activity["in_events"] + activity["out_events"]
    return activity


temporal_activity = build_temporal_edge_events(edges_df, node_time)
temporal_activity.head()

In [ ]:
# считаем burst-признаки активности по временным шагам

# вход: узлы, temporal activity и граф; выход: node-level burst-признаки и rule-based score.
def build_burst_features(nodes, temporal_activity, graph, feature_columns):
    # local_feature_cols дает proxy объема транзакции из анонимизированных признаков elliptic.
    local_feature_cols = feature_columns[:94] if len(feature_columns) >= 94 else feature_columns

    base = nodes[["txId", "time_step", "label", "class_name"]].copy()
    base = base.merge(
        temporal_activity,
        left_on=["txId", "time_step"],
        right_on=["txId", "event_time"],
        how="left",
    )
    base[["in_events", "out_events", "total_events"]] = base[
        ["in_events", "out_events", "total_events"]
    ].fillna(0)
    base = base.drop(columns=["event_time"])

    degree_cols = fan_features[["txId", "in_degree", "out_degree"]].copy()
    degree_cols["degree"] = degree_cols["in_degree"] + degree_cols["out_degree"]
    base = base.merge(degree_cols, on="txId", how="left")

    feature_volume = nodes[["txId"]].copy()
    feature_volume["feature_volume_proxy"] = nodes[local_feature_cols].abs().mean(axis=1)
    base = base.merge(feature_volume, on="txId", how="left")

    # neighbor_activity переносит burst-сигнал с локального окружения узла, а не только с самого txId.
    degree_map = base.set_index("txId")["degree"].to_dict()
    neighbor_activity = {}
    for node in base["txId"]:
        if node not in graph:
            neighbor_activity[node] = 0
            continue
        predecessors = list(graph.predecessors(node))
        successors = list(graph.successors(node))
        neighbors = set(predecessors + successors)
        neighbor_activity[node] = sum(degree_map.get(neighbor, 0) for neighbor in neighbors)
    base["neighbor_degree_sum"] = base["txId"].map(neighbor_activity).fillna(0)

    # zscore_cols нормируются внутри timestep, чтобы признаки отражали всплеск относительно текущего времени.
    zscore_cols = [
        "in_events",
        "out_events",
        "total_events",
        "degree",
        "neighbor_degree_sum",
        "feature_volume_proxy",
    ]
    for col in zscore_cols:
        base[f"{col}_z"] = base.groupby("time_step")[col].transform(
            lambda values: robust_zscore(values.to_numpy(dtype=float))
        )

    score_cols = [
        "total_events_z",
        "degree_z",
        "neighbor_degree_sum_z",
        "feature_volume_proxy_z",
    ]
    for col in score_cols:
        base[f"{col}_norm"] = minmax(base[col].clip(lower=0))

    # burst_score_rule объединяет активность узла, степень, соседей и proxy-объем в один ранжирующий score.
    base["burst_score_rule"] = (
        0.35 * base["total_events_z_norm"]
        + 0.25 * base["degree_z_norm"]
        + 0.25 * base["neighbor_degree_sum_z_norm"]
        + 0.15 * base["feature_volume_proxy_z_norm"]
    )
    return base


burst_features = build_burst_features(nodes_df, temporal_activity, G, feature_cols)
print(f"Burst feature rows: {len(burst_features)}")
burst_features.sort_values("burst_score_rule", ascending=False).head()

In [ ]:
# обучаем Isolation Forest для unsupervised burst anomaly score
# iforest_cols содержит независимые burst-сигналы для unsupervised anomaly detector.
iforest_cols = [
    "in_events_z",
    "out_events_z",
    "total_events_z",
    "degree_z",
    "neighbor_degree_sum_z",
    "feature_volume_proxy_z",
]

# iforest_input стандартизирует признаки, чтобы detector не зависел от масштаба отдельных колонок.
iforest_frame = burst_features[iforest_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
iforest_input = StandardScaler().fit_transform(iforest_frame)

iforest = IsolationForest(
    n_estimators=300,
    contamination=0.02,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iforest.fit(iforest_input)

burst_features["burst_score_iforest"] = -iforest.decision_function(iforest_input)
burst_features["is_burst_iforest"] = (iforest.predict(iforest_input) == -1).astype(int)
burst_features["burst_score_iforest_norm"] = minmax(burst_features["burst_score_iforest"])

burst_rule_metrics = evaluate_ranking(burst_features, "burst_score_rule")
burst_iforest_metrics = evaluate_ranking(burst_features, "burst_score_iforest_norm")

pd.concat([burst_rule_metrics, burst_iforest_metrics], ignore_index=True)

In [ ]:
# включаем опциональный GRU-блок для burst forecasting error
RUN_GRU_BURST = True


# вход: события активности; выход: матрица txId x timestep для прогноза временного ряда.
def build_node_activity_matrix(activity, time_steps, top_nodes=2000):
    # dynamic_nodes ограничивает gru-блок самыми активными узлами, чтобы обучение было быстрым.
    dynamic_nodes = (
        activity.groupby("txId")["total_events"]
        .sum()
        .sort_values(ascending=False)
        .head(top_nodes)
        .index
    )
    matrix = (
        activity[activity["txId"].isin(dynamic_nodes)]
        .pivot_table(
            index="txId",
            columns="event_time",
            values="total_events",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(columns=sorted(time_steps), fill_value=0)
    )
    return matrix


activity_matrix = build_node_activity_matrix(
    temporal_activity,
    nodes_df["time_step"].unique(),
    top_nodes=2000,
)
print(f"Activity matrix shape: {activity_matrix.shape}")

if not RUN_GRU_BURST or torch is None or activity_matrix.empty:
    burst_features["burst_score_gru_error_norm"] = 0.0
    print("GRU burst section skipped")
else:
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    # модель GRUForecaster учится предсказывать следующий timestep активности узла.
    class GRUForecaster(nn.Module):
        def __init__(self, hidden_size=16):
            super().__init__()
            self.gru = nn.GRU(input_size=1, hidden_size=hidden_size, batch_first=True)
            self.head = nn.Linear(hidden_size, 1)

        def forward(self, x):
            output, _ = self.gru(x)
            return self.head(output)

    # values логарифмируются, чтобы редкие всплески не задавили всю ошибку прогноза.
    values = np.log1p(activity_matrix.to_numpy(dtype=np.float32))
    train_x = torch.tensor(values[:, :-1, None], dtype=torch.float32)
    train_y = torch.tensor(values[:, 1:, None], dtype=torch.float32)
    loader = DataLoader(TensorDataset(train_x, train_y), batch_size=128, shuffle=True)

    model = GRUForecaster(hidden_size=16).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(20):
        losses = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(batch_x)
            loss = loss_fn(prediction, batch_y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if epoch in {0, 9, 19}:
            print(f"GRU epoch {epoch + 1}: loss {np.mean(losses):.6f}")

    model.eval()
    with torch.no_grad():
        prediction = model(train_x.to(DEVICE)).cpu().numpy().squeeze(-1)
    # errors становятся burst-признаком: нетипичный ряд активности хуже предсказывается gru.
    errors = ((prediction - values[:, 1:]) ** 2).mean(axis=1)

    gru_scores = pd.DataFrame({
        "txId": activity_matrix.index,
        "burst_score_gru_error": errors,
    })
    burst_features = burst_features.merge(gru_scores, on="txId", how="left")
    burst_features["burst_score_gru_error"] = burst_features[
        "burst_score_gru_error"
    ].fillna(0)
    burst_features["burst_score_gru_error_norm"] = minmax(
        burst_features["burst_score_gru_error"]
    )

# 4. Подготовка enriched Elliptic Data Set

В этом разделе формируется единая таблица признаков.

Логика:

1. Исходные признаки Elliptic остаются без изменений.
2. Fan-признаки добавляются по `txId`.
3. Peel-признаки добавляются по `txId`.
4. Burst-признаки добавляются по `txId` и `time_step`.
5. Пропуски в pattern-признаках заполняются нулями.
6. Итоговые CSV сохраняются в `outputs/elliptic_enriched_dataset`.

In [ ]:
# задаем списки признаков для финального объединения
# fan_feature_cols, peel_feature_cols и burst_feature_cols задают контракт экспортируемых pattern-признаков.
fan_feature_cols = [
    "txId",
    "in_degree",
    "out_degree",
    "z_in",
    "z_out",
    "hub_type",
    "is_fanin_hub",
    "is_fanout_hub",
    "is_bipartite_hub",
    "anomalousness_score",
    "elliptic_time_step_feature",
]

peel_feature_cols = [
    "txId",
    "peel_in_chain_any",
    "peel_chain_count",
    "peel_baseline_chain_participation",
    "peel_start_count",
    "peel_middle_count",
    "peel_end_count",
    "peel_max_chain_length",
    "peel_mean_ae_error",
    "peel_max_ae_error",
    "peel_mean_tab_anom_score",
    "peel_mean_gnn_anom_score",
]

burst_feature_cols = [
    "txId",
    "time_step",
    "in_events",
    "out_events",
    "total_events",
    "in_events_z",
    "out_events_z",
    "total_events_z",
    "degree_z",
    "neighbor_degree_sum_z",
    "feature_volume_proxy_z",
    "burst_score_rule",
    "burst_score_iforest_norm",
    "is_burst_iforest",
    "burst_score_gru_error_norm",
]

fan_features_export = fan_features[fan_feature_cols].copy()
peel_features_export = peel_features[peel_feature_cols].copy()
burst_features_export = burst_features[burst_feature_cols].copy()

print(f"Fan features: {fan_features_export.shape}")
print(f"Peel features: {peel_features_export.shape}")
print(f"Burst features: {burst_features_export.shape}")

In [ ]:
# объединяем исходные признаки с fan, peel и burst признаками
enriched_df = features_df.copy()
enriched_df = enriched_df.merge(classes_df, on="txId", how="left")
enriched_df = enriched_df.merge(fan_features_export, on="txId", how="left")
enriched_df = enriched_df.merge(peel_features_export, on="txId", how="left")
enriched_df = enriched_df.merge(
    burst_features_export,
    on=["txId", "time_step"],
    how="left",
)

# pattern_numeric_cols выделяет только числовые новые признаки, где пропуски можно безопасно заменить нулями.
pattern_numeric_cols = [
    col for col in enriched_df.columns
    if col not in {"txId", "class", "class_name", "hub_type"}
    and (
        col.startswith("is_")
        or col.startswith("z_")
        or col.startswith("peel_")
        or col.startswith("burst_")
        or col.endswith("_events")
        or col.endswith("_events_z")
        or col in {
            "in_degree",
            "out_degree",
            "anomalousness_score",
            "elliptic_time_step_feature",
            "degree_z",
            "neighbor_degree_sum_z",
            "feature_volume_proxy_z",
        }
    )
]

enriched_df[pattern_numeric_cols] = enriched_df[pattern_numeric_cols].fillna(0)
enriched_df["hub_type"] = enriched_df["hub_type"].fillna("normal")

# hub_type_code переводит категориальный тип hub в числовой признак для ml-моделей.
enriched_df["hub_type_code"] = enriched_df["hub_type"].map({
    "normal": 0,
    "fanin": 1,
    "fanout": 2,
    "bipartite": 3,
}).fillna(0).astype(int)

print(f"Original features shape: {features_df.shape}")
print(f"Enriched shape: {enriched_df.shape}")
print(f"Added columns: {enriched_df.shape[1] - features_df.shape[1]}")

enriched_df.head()

In [ ]:
# формируем каталог признаков по группам
# feature_groups нужен как каталог: какие колонки пришли из base, fan, peel и burst блоков.
feature_groups = {
    "base_features": feature_cols,
    "fan_features": [col for col in fan_feature_cols if col != "txId"],
    "peel_features": [col for col in peel_feature_cols if col != "txId"],
    "burst_features": [col for col in burst_feature_cols if col not in {"txId", "time_step"}],
    "target_columns": ["class", "label", "class_name"],
}

feature_catalog = []
for group_name, columns in feature_groups.items():
    for col in columns:
        feature_catalog.append({"feature_group": group_name, "column": col})
feature_catalog_df = pd.DataFrame(feature_catalog)

feature_catalog_df.groupby("feature_group").size().reset_index(name="n_columns")

In [ ]:
# сохраняем отдельные признаки и итоговый enriched dataset
fan_features_export.to_csv(OUTPUT_DIR / "fan_features.csv", index=False)
peel_features_export.to_csv(OUTPUT_DIR / "peel_features.csv", index=False)
burst_features_export.to_csv(OUTPUT_DIR / "burst_features.csv", index=False)
peel_chain_scores.to_csv(OUTPUT_DIR / "peel_chain_scores.csv", index=False)
feature_catalog_df.to_csv(OUTPUT_DIR / "feature_catalog.csv", index=False)

enriched_df.to_csv(OUTPUT_DIR / "elliptic_enriched_features_with_classes.csv", index=False)

# вариант без текстовых категорий удобнее для ml и загрузки как kaggle dataset.
ml_ready_df = enriched_df.drop(columns=["hub_type", "class_name"])
ml_ready_df.to_csv(OUTPUT_DIR / "elliptic_enriched_ml_ready.csv", index=False)

print(f"Saved fan features: {OUTPUT_DIR / 'fan_features.csv'}")
print(f"Saved peel features: {OUTPUT_DIR / 'peel_features.csv'}")
print(f"Saved burst features: {OUTPUT_DIR / 'burst_features.csv'}")
print(f"Saved enriched dataset: {OUTPUT_DIR / 'elliptic_enriched_features_with_classes.csv'}")
print(f"Saved ML-ready dataset: {OUTPUT_DIR / 'elliptic_enriched_ml_ready.csv'}")

## Итог

На выходе формируются файлы:

- `fan_features.csv` — признаки Fan-in / Fan-out;
- `peel_features.csv` — node-level признаки Peel Chains;
- `burst_features.csv` — признаки Burst Patterns;
- `peel_chain_scores.csv` — chain-level scores для анализа цепочек;
- `feature_catalog.csv` — каталог групп признаков;
- `elliptic_enriched_features_with_classes.csv` — расширенный датасет с классами и текстовыми категориями;
- `elliptic_enriched_ml_ready.csv` — расширенный ML-ready датасет без текстовых категорий.

Для дальнейшей классификации лучше начинать с `elliptic_enriched_ml_ready.csv` и делать ablation:

1. исходные признаки Elliptic;
2. исходные признаки + Fan-in/Fan-out;
3. исходные признаки + Fan-in/Fan-out + Peel Chains;
4. исходные признаки + все pattern-признаки.